# Graph plots

All charts are generated from the result files written by the pipeline. No numbers are typed in by hand.

- `results/retrieval_summary.csv`: one row per retrieval run (written by `05_retrieval_and_llm_setup`)
- `results/generation_summary.csv`: one row per generation run (written by `07_evaluation_and_analysis`)

Figures are saved to `results/figures/`.

In [18]:
from IPython.display import HTML, display

# Show full outputs (no scroll boxes)
display(HTML("""
<style>
div.output_scroll { height: auto !important; max-height: none !important; overflow-y: visible !important; }
.jp-OutputArea-output { max-height: none !important; }
</style>
"""))

In [19]:
%run -i "00_config.ipynb"

Project root: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag_project_cpuonly
CFG_NAME: chunk_w512_ov0.25
PASSAGES_PATH: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag_project_cpuonly\data\processed\passages.parquet
Q_PATH: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag_project_cpuonly\data\processed\questions_eval.parquet
EMB_DIR: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag_project_cpuonly\embeddings\intfloat_e5-small-v2\chunk_w512_ov0.25
RESULTS_DIR: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag_project_cpuonly\results
MODEL_PATH: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag_project_cpuonly\models\Qwen2.5-3B-Instruct-Q4_K_M.gguf | exists: True


## Settings

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

K = 5                 # which top-k to plot (must be one of K_LIST in notebook 05)
DEFAULT_EF = 96       # efSearch used for Dense/Hybrid in the main experiments
OVERLAP_CHUNK = 64    # chunk size used for the overlap experiment
COMPARE_CHUNK = 512   # chunk size for the BM25 vs Dense vs Hybrid bar chart
HNSW_CHUNK = 512      # chunk size used for the HNSW (efSearch) experiment
MAIN_OVERLAP = OVERLAP_RATIO   # overlap used for all charts except the overlap chart

# efSearch values behind "Low / Medium / High"; change these to the values you actually used
EF_SETTINGS = {"Low": 16, "Medium": 96, "High": 256}

# Short display names for the generator files (key = GGUF file name without .gguf)
MODEL_LABELS = {
    "Qwen2.5-3B-Instruct-Q4_K_M": "Qwen 3B",
    "Qwen2.5-7B-Instruct-Q4_K_M": "Qwen 7B",
    "Llama-3.2-3B-Instruct-Q4_K_M": "LLaMA",   # ← change the key to your exact LLaMA file name (without .gguf)
}

RETRIEVERS = ["BM25", "Dense", "Hybrid"]
COLORS = {"BM25": "tab:blue", "Dense": "tab:orange", "Hybrid": "tab:green"}

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## Load results

In [21]:
ret = pd.read_csv(RETRIEVAL_SUMMARY_CSV)

if GENERATION_SUMMARY_CSV.exists():
    gen = pd.read_csv(GENERATION_SUMMARY_CSV)
else:
    print("No generation results yet (run notebook 07). EM/F1 charts will be skipped.")
    gen = pd.DataFrame(columns=["timestamp", "retriever", "generator", "chunk_size",
                                "overlap_ratio", "n_questions", "EM", "F1"])

# Older runs were saved without efSearch: Dense/Hybrid then used the default (96)
if "ef_search" not in ret.columns:
    ret["ef_search"] = np.nan
is_hnsw = ret["retriever"] != "BM25"
ret.loc[is_hnsw, "ef_search"] = ret.loc[is_hnsw, "ef_search"].fillna(DEFAULT_EF)

# If the same configuration was run more than once, keep only the latest run
ret = (ret.sort_values("timestamp")
          .drop_duplicates(["retriever", "chunk_size", "overlap_ratio", "ef_search"], keep="last"))
gen = (gen.sort_values("timestamp")
          .drop_duplicates(["retriever", "generator", "chunk_size", "overlap_ratio"], keep="last"))

# Short column names for the chosen K
ret = ret.rename(columns={
    f"Recall@{K}": "recall",
    f"MRR@{K}": "mrr",
    f"mean_retrieval_latency_ms@{K}": "latency_ms",
    f"mean_mem_peak_mb@{K}": "memory_mb",
})
gen["model"] = gen["generator"].map(MODEL_LABELS).fillna(gen["generator"])

def ret_at(chunk=None, overlap=None, ef=DEFAULT_EF):
    """Filter retrieval results. BM25 has no efSearch, so it is always kept."""
    d = ret
    if chunk is not None:
        d = d[d["chunk_size"] == chunk]
    if overlap is not None:
        d = d[np.isclose(d["overlap_ratio"], overlap)]
    if ef is not None:
        d = d[(d["retriever"] == "BM25") | (d["ef_search"] == ef)]
    return d

def enough(d, col, need=2, what="this chart"):
    """True if d has at least `need` distinct values of col; otherwise explain what to run."""
    n = d[col].nunique() if len(d) else 0
    if n < need:
        print(f"Skipping {what}: needs results for at least {need} different {col} values, found {n}. "
              f"Run more configurations (see the coverage table above).")
        return False
    return True

print(f"Retrieval runs: {len(ret)}   Generation runs: {len(gen)}")
print("\nCoverage: which configurations exist so far")
display(ret.pivot_table(index=["chunk_size", "overlap_ratio"], columns="retriever",
                        values="recall", aggfunc="size", fill_value=0))
display(ret[["retriever", "chunk_size", "overlap_ratio", "ef_search", "recall", "mrr", "latency_ms", "memory_mb"]]
        .sort_values(["retriever", "chunk_size", "overlap_ratio"]).round(3))

Retrieval runs: 3   Generation runs: 3

Coverage: which configurations exist so far


,retriever,BM25,Dense,Hybrid
chunk_size,overlap_ratio,,,
512,0.25,1,1,1


,retriever,chunk_size,overlap_ratio,ef_search,recall,mrr,latency_ms,memory_mb
0,BM25,512,0.25,NaN,0.842,0.749,21.199,4.253
1,Dense,512,0.25,96.0,0.745,0.612,91.619,1.794
2,Hybrid,512,0.25,96.0,0.783,0.654,122.386,1.832


## 1. Chunk size: all retrievers side by side (overlap 0)

In [22]:
d = ret_at(overlap=MAIN_OVERLAP)
if enough(d, "chunk_size", what=f"chunk-size chart (overlap {MAIN_OVERLAP})"):
  metrics = [("recall", "Recall@%d (↑)" % K), ("mrr", "MRR@%d (↑)" % K),
             ("latency_ms", "Latency ms (↓)"), ("memory_mb", "Peak memory MB (↓)")]
  
  fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
  for ax, (col, title) in zip(axes.ravel(), metrics):
      for r in RETRIEVERS:
          g = d[d["retriever"] == r].sort_values("chunk_size")
          ax.plot(g["chunk_size"], g[col], marker="o", color=COLORS[r], label=r)
      ax.set_title(title)
      ax.set_xlabel("Chunk size")
      ax.set_xscale("log", base=2)
      ax.set_xticks(sorted(d["chunk_size"].unique()))
      ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
      ax.grid(alpha=0.3)
  axes[0, 0].legend()
  fig.suptitle("Retrieval trade-offs across chunk sizes")
  plt.tight_layout()
  save(fig, "chunk_size_all_retrievers")
  plt.show()

Skipping chunk-size chart (overlap 0.0): needs results for at least 2 different chunk_size values, found 0. Run more configurations (see the coverage table above).


## 2. Chunk size: one trade-off chart per retriever

In [23]:
def plot_tradeoff(retriever):
    g = ret_at(overlap=MAIN_OVERLAP)
    g = g[g["retriever"] == retriever].sort_values("chunk_size")
    x = g["chunk_size"].astype(int).astype(str)   # evenly spaced categories

    fig, ax1 = plt.subplots(figsize=(12, 5))
    l1, = ax1.plot(x, g["recall"], marker="o", color="tab:blue", linewidth=2)
    l2, = ax1.plot(x, g["mrr"], marker="o", color="tab:orange", linewidth=2)
    ax1.set_xlabel("Chunk size")
    ax1.set_ylabel("Effectiveness (↑)")
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    l3, = ax2.plot(x, g["latency_ms"], marker="o", color="tab:green", linewidth=2)
    l4, = ax2.plot(x, g["memory_mb"], marker="o", color="tab:red", linewidth=2)
    ax2.set_ylabel("Efficiency cost (↓, log scale)")
    ax2.set_yscale("log")

    ax1.legend([l1, l2, l3, l4], [f"Recall@{K}", f"MRR@{K}", "Latency (ms)", "Memory (MB)"],
               loc="upper left", bbox_to_anchor=(1.08, 1.0))
    ax1.set_title(f"{retriever} retriever: trade-offs across chunk sizes")
    plt.tight_layout()
    save(fig, f"tradeoff_{retriever.lower()}")
    plt.show()

if enough(ret_at(overlap=MAIN_OVERLAP), "chunk_size", what="per-retriever trade-off charts"):
    for r in RETRIEVERS:
        plot_tradeoff(r)

Skipping per-retriever trade-off charts: needs results for at least 2 different chunk_size values, found 0. Run more configurations (see the coverage table above).


## 3. Bubble plots: quality vs latency (bubble size = memory)

In [24]:
def bubble_plot(y_metric, size_metric, title, bubble_scale=1200):
    d = ret_at(overlap=MAIN_OVERLAP)
    fig, ax = plt.subplots(figsize=(11, 7))
    max_size = d[size_metric].max()
    for r in RETRIEVERS:
        sub = d[d["retriever"] == r].sort_values("chunk_size")
        ax.scatter(sub["latency_ms"], sub[y_metric],
                   s=(sub[size_metric] / max_size) * bubble_scale,
                   alpha=0.6, color=COLORS[r], edgecolors="black", linewidths=0.5, label=r)
        for _, row in sub.iterrows():
            ax.annotate(f"{int(row['chunk_size'])}", (row["latency_ms"], row[y_metric]),
                        textcoords="offset points", xytext=(7, 7), fontsize=9, fontweight="bold")
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel(y_metric)
    ax.set_title(title + "\n(labels = chunk size)")
    ax.grid(alpha=0.35)
    ax.legend(title="Retriever", loc="center left", bbox_to_anchor=(1.02, 0.5), markerscale=0.4)
    plt.tight_layout()
    save(fig, f"bubble_{y_metric}_vs_latency")
    plt.show()

if enough(ret_at(overlap=MAIN_OVERLAP), "retriever", need=1, what="bubble plots"):
  bubble_plot("recall", "memory_mb", f"Recall@{K} vs latency (bubble size = memory)")
  bubble_plot("mrr", "memory_mb", f"MRR@{K} vs latency (bubble size = memory)")
  bubble_plot("memory_mb", "recall", "Memory vs latency (bubble size = recall)")

Skipping bubble plots: needs results for at least 1 different retriever values, found 0. Run more configurations (see the coverage table above).


## 4. HNSW settings (efSearch): Dense vs Hybrid heatmaps

In [25]:
hm_metrics = [("recall", "Recall"), ("mrr", "MRR"), ("latency_ms", "Latency (ms)"), ("memory_mb", "Memory (MB)")]
settings = list(EF_SETTINGS)

def hnsw_matrix(retriever):
    d = ret_at(chunk=HNSW_CHUNK, overlap=MAIN_OVERLAP, ef=None)
    d = d[d["retriever"] == retriever].set_index("ef_search")
    missing = [ef for ef in EF_SETTINGS.values() if ef not in d.index]
    if missing:
        print(f"{retriever}: no results for efSearch {missing} at chunk {HNSW_CHUNK}; run them in notebook 05 first.")
        return None
    return np.array([[d.loc[ef, col] for col, _ in hm_metrics] for ef in EF_SETTINGS.values()], dtype=float)

def plot_heatmap(title, raw, fname):
    mins, maxs = raw.min(axis=0), raw.max(axis=0)
    scaled = (raw - mins) / np.where(maxs - mins == 0, 1, maxs - mins)
    fig, ax = plt.subplots(figsize=(8.6, 4.7))
    im = ax.imshow(scaled.T, aspect="auto", cmap="plasma")
    ax.set_xticks(range(len(settings)))
    ax.set_xticklabels([f"{s}\n(ef={EF_SETTINGS[s]})" for s in settings])
    ax.set_yticks(range(len(hm_metrics)))
    ax.set_yticklabels([label for _, label in hm_metrics])
    for i in range(len(hm_metrics)):
        for j in range(len(settings)):
            ax.text(j, i, f"{raw[j, i]:.2f}", ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.85))
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Relative value (per-metric min–max)")
    plt.tight_layout()
    save(fig, fname)
    plt.show()

for r in ["Dense", "Hybrid"]:
    m = hnsw_matrix(r)
    if m is not None:
        plot_heatmap(f"{r} retriever: metrics across HNSW settings (chunk {HNSW_CHUNK})", m, f"hnsw_heatmap_{r.lower()}")

Dense: no results for efSearch [16, 96, 256] at chunk 512; run them in notebook 05 first.
Hybrid: no results for efSearch [16, 96, 256] at chunk 512; run them in notebook 05 first.


## 5. BM25 vs Dense vs Hybrid at one chunk size

In [26]:
def plot_compare():
    d = ret_at(chunk=COMPARE_CHUNK, overlap=MAIN_OVERLAP).set_index("retriever").reindex(RETRIEVERS)
    x = np.arange(len(RETRIEVERS))
    w = 0.35

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax1.bar(x - w/2, d["recall"], w, label=f"Recall@{K}")
    ax1.bar(x + w/2, d["mrr"], w, label=f"MRR@{K}")
    ax1.set_ylim(0, 1)
    ax1.set_title(f"Retrieval quality (chunk {COMPARE_CHUNK})")

    ax2.bar(x - w/2, d["latency_ms"], w, label="Latency (ms)")
    ax2.bar(x + w/2, d["memory_mb"], w, label="Memory (MB)")
    ax2.set_title(f"Efficiency (chunk {COMPARE_CHUNK})")

    for ax in (ax1, ax2):
        ax.set_xticks(x)
        ax.set_xticklabels(RETRIEVERS)
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        ax.legend()
    plt.tight_layout()
    save(fig, f"retriever_comparison_chunk{COMPARE_CHUNK}")
    plt.show()

d5 = ret_at(chunk=COMPARE_CHUNK, overlap=MAIN_OVERLAP)
if d5.empty:
    print(f"Skipping: no results at chunk {COMPARE_CHUNK}, overlap {MAIN_OVERLAP}. Change COMPARE_CHUNK / MAIN_OVERLAP in Settings.")
else:
    plot_compare()

Skipping: no results at chunk 512, overlap 0.0. Change COMPARE_CHUNK / MAIN_OVERLAP in Settings.


## 6. Overlap ratio: latency and memory per retriever

In [27]:
def plot_overlap():
    d = ret_at(chunk=OVERLAP_CHUNK)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax1, r in zip(axes, RETRIEVERS):
        g = d[d["retriever"] == r].sort_values("overlap_ratio")
        pct = g["overlap_ratio"] * 100
        l1, = ax1.plot(pct, g["latency_ms"], marker="o", linewidth=2, color="tab:blue")
        ax1.set_xlabel("Overlap ratio (%)")
        ax1.set_ylabel("Latency (ms)")
        ax1.set_xticks(pct)
        ax1.grid(alpha=0.35)
        ax2 = ax1.twinx()
        l2, = ax2.plot(pct, g["memory_mb"], marker="o", linewidth=2, color="tab:orange")
        ax2.set_ylabel("Memory (MB)")
        ax1.legend([l1, l2], ["Latency (ms)", "Memory (MB)"], loc="upper left")
        ax1.set_title(f"{r} (chunk {OVERLAP_CHUNK})")
    fig.suptitle("Latency & memory vs overlap ratio")
    plt.tight_layout()
    save(fig, "overlap_latency_memory")
    plt.show()

if enough(ret_at(chunk=OVERLAP_CHUNK), "overlap_ratio", what=f"overlap chart (chunk {OVERLAP_CHUNK})"):
    plot_overlap()

Skipping overlap chart (chunk 64): needs results for at least 2 different overlap_ratio values, found 0. Run more configurations (see the coverage table above).


## 7. Generation quality: EM and F1 per retriever and model

In [28]:
models = [m for m in MODEL_LABELS.values() if m in set(gen["model"])] + \
         sorted(set(gen["model"]) - set(MODEL_LABELS.values()))

def plot_em_f1(chunk):
    d = gen[(gen["chunk_size"] == chunk) & np.isclose(gen["overlap_ratio"], MAIN_OVERLAP)]
    if d.empty:
        return
    n = int(d["n_questions"].min())
    x = np.arange(len(RETRIEVERS))
    w = 0.8 / len(models)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), dpi=130)
    fig.subplots_adjust(top=0.70, wspace=0.30)
    for ax, metric in zip(axes, ["EM", "F1"]):
        pivot = d.pivot(index="retriever", columns="model", values=metric).reindex(index=RETRIEVERS, columns=models)
        for i, m in enumerate(models):
            ax.bar(x + (i - (len(models) - 1) / 2) * w, pivot[m].values, w, label=m)
        ax.set_xticks(x)
        ax.set_xticklabels(RETRIEVERS)
        ax.set_ylim(0, max(0.6, float(np.nanmax(pivot.values)) + 0.05))
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.grid(axis="y", alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(models), frameon=False, bbox_to_anchor=(0.5, 0.90))
    fig.suptitle(f"Chunk size = {chunk}   (n = {n} questions)", fontsize=12, y=0.97)
    save(fig, f"em_f1_chunk{chunk}")
    plt.show()

if gen.empty:
    print("No generation results yet.")
elif gen[np.isclose(gen["overlap_ratio"].astype(float), MAIN_OVERLAP)].empty:
    print(f"No generation results at overlap {MAIN_OVERLAP}. Change MAIN_OVERLAP in Settings or run that config.")
for chunk in sorted(gen["chunk_size"].unique()):
    plot_em_f1(chunk)

No generation results at overlap 0.0. Change MAIN_OVERLAP in Settings or run that config.


## 8. Best configurations

In [29]:
best = (gen.sort_values(["EM", "F1"], ascending=False)
           [["retriever", "model", "chunk_size", "overlap_ratio", "n_questions", "EM", "F1"]]
           .head(10).round(3))
display(best)

,retriever,model,chunk_size,overlap_ratio,n_questions,EM,F1
1,Dense,Qwen 3B,512,0.25,50,0.42,0.498
0,BM25,Qwen 3B,512,0.25,50,0.40,0.469
2,Hybrid,Qwen 3B,512,0.25,50,0.36,0.445
